In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from pathlib import Path

input_root = '/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026'
json_path = os.path.join(input_root, 'train', 'annotations_coco.json')

with open(json_path, 'r') as f:
    coco_data = json.load(f)

images_df = pd.DataFrame(coco_data['images'])
anns_df = pd.DataFrame(coco_data['annotations'])

# Відфільтруємо ігноровані зони (iscrowd = 1) для аналізу
valid_anns_df = anns_df[anns_df['iscrowd'] == 0].copy()

# 2. EDA: Розподіл розмірів BBox
valid_anns_df[['x', 'y', 'w', 'h']] = pd.DataFrame(valid_anns_df['bbox'].tolist(), index=valid_anns_df.index)

plt.figure(figsize=(8, 6))
plt.scatter(valid_anns_df['w'], valid_anns_df['h'], alpha=0.1, s=1)
plt.title("Розподіл розмірів об'єктів (w vs h)")
plt.xlabel("Ширина (px)")
plt.ylabel("Висота (px)")
plt.axvline(x=30, color='r', linestyle='--', label='30 px')
plt.axhline(y=30, color='r', linestyle='--')
plt.legend()
plt.grid(True)
plt.show()

# 3. Валідація: Sequence-level split (GroupKFold)
images_df['clip'] = images_df['file_name'].apply(lambda x: x.split('/')[1])

gkf = GroupKFold(n_splits=5)
images_df['fold'] = -1

for fold, (train_idx, val_idx) in enumerate(gkf.split(X=images_df, groups=images_df['clip'])):
    images_df.loc[val_idx, 'fold'] = fold

print("Кількість унікальних відео у кожному фолді:")
print(images_df.groupby('fold')['clip'].nunique())

# Зберігаємо розбиття у робочу папку Kaggle
images_df.to_csv('/kaggle/working/cv_splits.csv', index=False)

In [ ]:
import cv2
from pathlib import Path
from tqdm.notebook import tqdm

def mask_ignore_regions(input_base, output_base, split_name):
    # Беремо картинки та ігнор-зони з read-only директорії Kaggle
    images_dir = Path(input_base) / split_name / "images"
    ignore_dir = Path(input_base) / split_name / "ignore"
    
    # Зберігаємо у папку з доступом до запису
    out_dir = Path(output_base) / split_name / "images_masked"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    clips = [d.name for d in images_dir.iterdir() if d.is_dir()]
    
    for clip in tqdm(clips, desc=f"Processing {split_name}"):
        (out_dir / clip).mkdir(parents=True, exist_ok=True)
        
        image_files = list((images_dir / clip).glob("*.jpg"))
        for img_path in image_files:
            frame_name = img_path.stem
            ignore_file = ignore_dir / clip / f"{frame_name}.txt"
            
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            h_img, w_img = img.shape[:2]
            
            if ignore_file.exists():
                with open(ignore_file, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 4:
                            cx, cy, w, h = map(float, parts[:4])
                            x1 = int((cx - w / 2) * w_img)
                            y1 = int((cy - h / 2) * h_img)
                            x2 = int((cx + w / 2) * w_img)
                            y2 = int((cy + h / 2) * h_img)
                            
                            cv2.rectangle(img, (x1, y1), (x2, y2), (128, 128, 128), -1)
            
            out_path = out_dir / clip / f"{frame_name}.jpg"
            cv2.imwrite(str(out_path), img)

# Шляхи Kaggle
input_root = '/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026'
output_root = "/kaggle/working/masked_dataset"

mask_ignore_regions(input_root, output_root, "train")
mask_ignore_regions(input_root, output_root, "val")
print("Маскування завершено!")

In [ ]:
import shutil
import yaml
import os


input_root = '/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026'
output_root = "/kaggle/working/masked_dataset"

# Копіюємо мітки з read-only директорії у нашу робочу папку
shutil.copytree(os.path.join(input_root, 'train/labels'), 
                os.path.join(output_root, 'train/labels_masked'), dirs_exist_ok=True)
shutil.copytree(os.path.join(input_root, 'val/labels'), 
                os.path.join(output_root, 'val/labels_masked'), dirs_exist_ok=True)

print("Мітки успішно скопійовані!")

# Генеруємо data_masked.yaml одразу в робочій папці
data_config = {
    'path': output_root, # Абсолютний шлях до згенерованого датасету в Kaggle
    'train': 'train/images_masked',
    'val': 'val/images_masked',
    'nc': 8,
    'names': {
        0: 'person', 1: 'bicycle', 2: 'car', 3: 'van', 
        4: 'truck', 5: 'tricycle', 6: 'bus', 7: 'motor'
    }
}

with open(os.path.join(output_root, 'data_masked.yaml'), 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)
    
print(f"data_masked.yaml створено в {output_root}")

In [ ]:
import os
import glob
import yaml

base_path = '/kaggle/working/masked_dataset'

# 1. Перейменовуємо папки на стандартні, які очікує YOLO
for split in ['train', 'val']:
    images_old = os.path.join(base_path, split, 'images_masked')
    images_new = os.path.join(base_path, split, 'images')
    if os.path.exists(images_old):
        os.rename(images_old, images_new)
        
    labels_old = os.path.join(base_path, split, 'labels_masked')
    labels_new = os.path.join(base_path, split, 'labels')
    if os.path.exists(labels_old):
        os.rename(labels_old, labels_new)

# 2. Знищуємо всі биті .cache файли (як той, що у твоїй помилці)
for cache_file in glob.glob(f'{base_path}/**/*.cache', recursive=True):
    os.remove(cache_file)
    print(f"Видалено старий кеш: {cache_file}")

# 3. Оновлюємо YAML
yaml_path = f'{base_path}/data_masked.yaml'
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['train'] = 'train/images'
cfg['val'] = 'val/images'

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print("Папки перейменовано, кеш очищено, YAML оновлено!")

In [ ]:
!pip install ultralytics

import os
os.environ["WANDB_MODE"] = "disabled"
from ultralytics import YOLO

yaml_path = '/kaggle/working/masked_dataset/data_masked.yaml'

model = YOLO('yolov8n.pt')
results = model.train(
    data=yaml_path,
    epochs=10,
    imgsz=1024,
    batch=16, 
    device=0, 
    project='/kaggle/working/ucu_ai_hw2', 
    name='yolov8n_baseline'
)

In [ ]:
import pandas as pd
from ultralytics import YOLO
import os

# 1. Шляхи
input_root = '/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026'
test_csv_path = os.path.join(input_root, 'test_images.csv')
model_path = '/kaggle/working/ucu_ai_hw2/yolov8n_baseline-2/weights/best.pt'

# 2. Завантажуємо навчену модель
model = YOLO(model_path)

# 3. Читаємо інформацію про тестові зображення
test_df = pd.read_csv(test_csv_path)

# Створюємо список для збереження результатів
submission_data = []

# 4. Проходимо по кожному зображенню
for index, row in test_df.iterrows():
    image_id = row['image_id']
    file_name = row['file_name']
    img_path = os.path.join(input_root, file_name)
    
    # Виконуємо передбачення (conf=0.25 - можна налаштувати)
    results = model.predict(source=img_path, conf=0.25, verbose=False)
    
    prediction_string = []
    
    # Отримуємо результати для першого (і єдиного) зображення в батчі
    result = results[0]
    
    # Якщо є знахідки
    if len(result.boxes) > 0:
        # ultralytics повертає координати xyxy (x1, y1, x2, y2)
        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()
        
        for i in range(len(boxes)):
            cls_id = int(classes[i])
            score = round(float(scores[i]), 2)
            
            # x1, y1 - це top-left. Вираховуємо ширину та висоту.
            x1, y1, x2, y2 = boxes[i]
            x_tl = int(round(x1))
            y_tl = int(round(y1))
            w = int(round(x2 - x1))
            h = int(round(y2 - y1))
            
            # Формат: class score x_top_left y_top_left width height
            pred_item = f"{cls_id} {score} {x_tl} {y_tl} {w} {h}"
            prediction_string.append(pred_item)
            
    # Об'єднуємо всі знахідки для зображення через пробіл
    pred_str_final = " ".join(prediction_string)
    submission_data.append({'image_id': image_id, 'PredictionString': pred_str_final})

# 5. Зберігаємо у CSV
submission_df = pd.DataFrame(submission_data)
submission_df.to_csv('submission.csv', index=False)
print("Файл submission.csv успішно згенеровано! Можеш завантажувати на Kaggle.")

In [ ]:
from ultralytics import RTDETR

yaml_path = '/kaggle/working/masked_dataset/data_masked.yaml'

model_rt = RTDETR('rtdetr-l.pt')
results_rt = model_rt.train(
    data=yaml_path,
    epochs=5,       # Зменшено для швидкості
    imgsz=640,      # Зменшено для трансформера
    batch=16, 
    device=0, 
    project='/kaggle/working/ucu_ai_hw2', 
    name='rtdetr_fast'
)

In [ ]:
import pandas as pd
from ultralytics import YOLO
from ultralytics import RTDETR
import os

input_root = '/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026'
test_csv_path = os.path.join(input_root, 'test_images.csv')

# Беремо найкращу модель (RT-DETR)
model_path = '/kaggle/working/ucu_ai_hw2/rtdetr_fast/weights/best.pt'
model = RTDETR(model_path)

test_df = pd.read_csv(test_csv_path)
submission_data = []

print("Починаємо інференс на тестовому датасеті...")

for index, row in test_df.iterrows():
    image_id = row['image_id']
    file_name = row['file_name']
    img_path = os.path.join(input_root, file_name)
    
    # Inference strategy: lower confidence to catch small objects
    results = model.predict(source=img_path, conf=0.20, verbose=False)
    
    prediction_string = []
    result = results[0]
    
    if len(result.boxes) > 0:
        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()
        
        for i in range(len(boxes)):
            cls_id = int(classes[i])
            score = round(float(scores[i]), 2)
            
            x1, y1, x2, y2 = boxes[i]
            x_tl = int(round(x1))
            y_tl = int(round(y1))
            w = int(round(x2 - x1))
            h = int(round(y2 - y1))
            
            pred_item = f"{cls_id} {score} {x_tl} {y_tl} {w} {h}"
            prediction_string.append(pred_item)
            
    pred_str_final = " ".join(prediction_string)
    submission_data.append({'image_id': image_id, 'PredictionString': pred_str_final})

submission_df = pd.DataFrame(submission_data)
submission_df.to_csv('submission_final.csv', index=False)
print("Файл submission_final.csv успішно згенеровано!")